In [ ]:
# TODO: add comments below Hierarchical Triangle Mesh section

# Global Ocean Triangle (Multi-)Mesh comparison

In this notebook different meshes for the data-driven model under development are explored. The first is the icosahedral multi-mesh used by GraphCast as is, the second is a masked version of the previous, finally, a hierarchical triangle mesh obtained with `seamsh` has been considered.

In [ ]:
import tempfile
import zipfile
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import requests
import seamsh.geometry
import xarray
from geopy.geocoders import Nominatim
from osgeo import osr
from seamsh.geometry import CurveType
import networkx as nx

from graphcast import icosahedral_mesh
from graphcast import mesh_connectivity
from graphcast import mesh_graph
from graphcast import model
from graphcast import ocean_mesh
from graphcast import plot_utils

In [ ]:
save_figures = True
save_mesh = True

output_dir = Path('outputs/mesh_comparison')
if not output_dir.exists():
  output_dir.mkdir(parents=True)

In [ ]:
# The zoom-ins will be around the following area.
geolocator = Nominatim(user_agent="mesh_comparison")
location = geolocator.geocode("Bryant University")
width, height = 65.0, 20.0
bounds = [location.longitude - width / 2, location.longitude + width / 2,
          location.latitude - height / 2, location.latitude + height / 2]

## Icosahedral Multi-Mesh

In [ ]:
# Icosahedral hierarchical meshes are 3-dimensional and defined on a unit sphere.
# They will be projected on the reference geoid via PROJ, using `mesh_to_wgs`
SPLITS = 6

mesh_hierarchy = icosahedral_mesh.get_hierarchy_of_triangular_meshes_for_sphere(splits=SPLITS)

In [ ]:
def hierarchy_to_nx(meshes):
    multimesh = mesh_graph.merge_meshes(meshes)
    wgs_graph = mesh_graph.mesh_to_wgs(multimesh)
    nx_graph = mesh_graph.wgs_to_nx(wgs_graph)
    nx_graph.add_edges_from(zip(*multimesh.edges))
    return nx_graph

In [ ]:
mesh_graph.graph_summary(hierarchy_to_nx(mesh_hierarchy))

In [ ]:
fig = plt.figure(dpi=300)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.InterruptedGoodeHomolosine(emphasis='ocean'))
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='grey'))

# Plot refinement levels separately, with thinner edges for finer meshes
for (split, mesh) in enumerate(mesh_hierarchy[:4]):

  wgs_graph = mesh_graph.mesh_to_wgs(mesh, unit_sphere=True)
  lc = plot_utils.get_line_collection(wgs_graph, linewidths=0.08 * (4 - split), colors="green", transform=ccrs.Geodetic())
  ax.add_collection(lc)
  latitudes, longitudes = wgs_graph.vertices
  ax.scatter(longitudes, latitudes, marker='.', color="green", s=(4 - split), zorder=2, transform=ccrs.PlateCarree())

# Add a rectangle patch around zoom-in
ax.add_patch(mpatches.Rectangle(xy=(location.longitude - width / 2, location.latitude - height / 2), width=width, height=height,
                                facecolor='none', edgecolor='k', zorder=3, transform=ccrs.PlateCarree()))
if save_figures:
    plt.savefig(output_dir / 'icosahedral_multimesh.png', bbox_inches='tight')

In [ ]:
fig = plt.figure(dpi=300)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mercator())
ax.set_extent(bounds)
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black'))
ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.4, color='k', linestyle='--')

# Plot refinement levels separately, with thinner edges for finer meshes
for (split, mesh) in enumerate(mesh_hierarchy):

  wgs_graph = mesh_graph.mesh_to_wgs(mesh, unit_sphere=True)
  lc = plot_utils.get_line_collection(wgs_graph, linewidths=0.08 * (7 - split), colors="green", transform=ccrs.Geodetic())
  ax.add_collection(lc)
  latitudes, longitudes = wgs_graph.vertices
  ax.scatter(longitudes, latitudes, marker='.', color="green", s=(7 - split), zorder=2, transform=ccrs.PlateCarree())

if save_figures:
    plt.savefig(output_dir / 'icosahedral_multimesh_zoomin.png', bbox_inches='tight')

In [ ]:
# Download mask from Copernicus Marine using provided download script and configuration file
seamask = xarray.open_zarr('data/dataset/glorys12_bathymetry_tres-static_res-1.0_levels-1.zip')
mask = seamask.mask
mask = mask.criterion()
mask = mask.rename(latitude="lat", longitude="lon")
mask

In [ ]:
# Multi-mesh masking is per refinement level, as it depends on `_get_max_edge_distance(mesh)`,
# here we select only the first 4 levels and se `RADIUS_QUERY_FRACTION_EDGE_LENGTH` accordingly.
RADIUS_QUERY_FRACTION_EDGE_LENGTH = 0.2

coarse_multimesh = mesh_graph.merge_meshes(mesh_hierarchy[:4])
connected_mesh_nodes = mesh_connectivity.get_connected_mesh_nodes(grid_lat=mask.lat,
                                                                  grid_lon=mask.lon,
                                                                  mesh_graph=coarse_multimesh,
                                                                  grid_mask=mask,
                                                                  query_radius=model._get_max_edge_distance(coarse_multimesh) * RADIUS_QUERY_FRACTION_EDGE_LENGTH)
masked_coarse_multimesh = mesh_connectivity.mask_multimesh(connected_mesh_vertices=connected_mesh_nodes, mesh=coarse_multimesh)

In [ ]:
fig = plt.figure(dpi=300)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.InterruptedGoodeHomolosine(emphasis='ocean'))
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black'))

# Masking operates on `MultiMeshGraph`, which is *not* a hierarchical mesh, as it stores faces only for the finest mesh.
# As for GraphCast, this piece of information is enough, because the decoder requires only them.
masked_coarse_wgs_graph = mesh_graph.mesh_to_wgs(masked_coarse_multimesh, unit_sphere=True)
lc = plot_utils.get_line_collection(masked_coarse_wgs_graph, linewidths=0.16, colors="green", transform=ccrs.Geodetic())
ax.add_collection(lc)
latitudes, longitudes = masked_coarse_wgs_graph.vertices
ax.scatter(longitudes, latitudes, marker='.', color="green", s=2.0, zorder=2, transform=ccrs.PlateCarree())

if save_figures:
    plt.savefig(output_dir / 'masked_icosahedral_multimesh.png', bbox_inches='tight')

In [ ]:
# Adjust `RADIUS_QUERY_FRACTION_EDGE_LENGTH` to work with 6 levels of refinement,
# i.e. the multi-mesh used in GraphCast paper.
RADIUS_QUERY_FRACTION_EDGE_LENGTH = 0.6

fine_multimesh = mesh_graph.merge_meshes(mesh_hierarchy)
connected_mesh_nodes = mesh_connectivity.get_connected_mesh_nodes(grid_lat=mask.lat,
                                                                  grid_lon=mask.lon,
                                                                  mesh_graph=fine_multimesh,
                                                                  grid_mask=mask,
                                                                  query_radius=model._get_max_edge_distance(fine_multimesh) * RADIUS_QUERY_FRACTION_EDGE_LENGTH)
masked_fine_multimesh = mesh_connectivity.mask_multimesh(connected_mesh_vertices=connected_mesh_nodes, mesh=fine_multimesh)

In [ ]:
masked_fine_multimesh.edges

In [ ]:
import networkx as nx
digraph = nx.DiGraph()
number_of_nodes = len(masked_fine_multimesh.vertices)
digraph.add_nodes_from(range(number_of_nodes))
digraph.add_edges_from(zip(*masked_fine_multimesh.edges))
# masked_fine_multimesh_wgs_graph = mesh_graph.mesh_to_wgs(masked_fine_multimesh)
# masked_fine_multimesh_nx_graph = mesh_graph.wgs_to_nx(masked_fine_multimesh_wgs_graph)
# masked_fine_multimesh_nx_graph.add_edges_from(zip(*masked_fine_multimesh.edges))
mesh_graph.graph_summary(digraph)

In [ ]:
fig = plt.figure(dpi=300)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mercator())
ax.set_extent(bounds)
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black'))
ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.4, color='k', linestyle='--')

masked_fine_wgs_graph = mesh_graph.mesh_to_wgs(masked_fine_multimesh, unit_sphere=True)
lc = plot_utils.get_line_collection(masked_fine_wgs_graph, linewidths=0.16, colors="green", transform=ccrs.Geodetic())
ax.add_collection(lc)
latitudes, longitudes = masked_fine_wgs_graph.vertices
ax.scatter(longitudes, latitudes, marker='.', color="green", s=2.0, zorder=2, transform=ccrs.PlateCarree())

if save_figures:
    plt.savefig(output_dir / 'masked_icosahedral_multimesh_zoomin.png', bbox_inches='tight')

In [ ]:
masked_fine_nx_graph = mesh_graph.wgs_to_nx(masked_fine_wgs_graph)
mesh_graph.graph_summary(masked_fine_nx_graph)

## Hierarchical Triangle Mesh

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
  req = requests.get("https://naciscdn.org/naturalearth/50m/physical/ne_50m_coastline.zip", allow_redirects=True)
  with tempfile.TemporaryFile(dir=tmpdir, suffix="zip") as file:
    file.write(req.content)
    with zipfile.ZipFile(file) as archive:
      archive.extractall(tmpdir)
  osr.UseExceptions()
  domain_srs = osr.SpatialReference()
  domain_srs.ImportFromProj4("+ellps=WGS84 +proj=stere +lat_0=90")
  domain = seamsh.geometry.Domain(domain_srs)
  domain.add_boundary_curves_shp(tmpdir + "/ne_50m_coastline.shp", "featurecla", CurveType.POLYLINE)

In [ ]:
# seamsh usage example taken from: https://jlambrechts.git-page.immc.ucl.ac.be/seamsh/examples/6-stereographics.html

SAMPLING = 10_000.0
EARTH_RADIUS = 6371008.7714

cart_srs = osr.SpatialReference()
cart_srs.ImportFromProj4("+ellps=WGS84 +proj=cart +units=m +x_0=0 +y_0=0")

dist_coast = seamsh.field.Distance(domain, SAMPLING, projection=cart_srs)

def mesh_size(distance_offset=20_000.0, min_size=20_000.0, max_size=200_000.0):

  def _mesh_size(x, projection):
    s_coast = np.clip(0.5 * (dist_coast(x, projection) - distance_offset), min_size, max_size)
    R_squared = EARTH_RADIUS ** 2
    x_squared = x ** 2
    stereo_factor = 2 / (1 + x_squared[:, 0] / R_squared + x_squared[:, 1] / R_squared)
    return s_coast / stereo_factor

  return _mesh_size

In [ ]:
COARSE_DOMAIN_MIN_SIZE = 20_000.0
COARSE_DOMAIN_MAX_SIZE = 200_000.0

coarse_domain = seamsh.geometry.coarsen_boundaries(domain, (0, 0), domain_srs, mesh_size(min_size=COARSE_DOMAIN_MIN_SIZE, max_size=COARSE_DOMAIN_MAX_SIZE))

In [ ]:
coarse_ocean_mesh = ocean_mesh_utils.get_ocean_mesh(domain=coarse_domain,
                                                    mesh_size=mesh_size,
                                                    save_mesh=(output_dir / 'coarse_ocean_mesh.msh') if save_mesh else None,
                                                    min_size=200_000.0,
                                                    max_size=800_000.0)
coarse_ocean_mesh_graph = mesh_graph.mesh_to_wgs(coarse_ocean_mesh)

In [ ]:
fig = plt.figure(dpi=300)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.InterruptedGoodeHomolosine(emphasis='ocean'))
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='grey'))
latitudes, longitudes = coarse_ocean_mesh_graph.vertices
ax.scatter(longitudes, latitudes, marker='.', color="green", s=2.0, zorder=2, alpha=0.0, transform=ccrs.PlateCarree())
lc = plot_utils.get_line_collection(coarse_ocean_mesh_graph, linewidths=0.16, colors="green", transform=ccrs.Geodetic())
ax.add_collection(lc)
ax.add_patch(mpatches.Rectangle(xy=(location.longitude - width / 2, location.latitude - height / 2), width=width, height=height,
                                facecolor='none', edgecolor='k', zorder=3, transform=ccrs.PlateCarree()))
if save_figures:
    plt.savefig(output_dir / 'coarse_ocean_mesh.png', bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(33.1, 46.8), dpi=450)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.InterruptedGoodeHomolosine(emphasis='ocean'))
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='grey'))
latitudes, longitudes = coarse_ocean_mesh_graph.vertices
ax.scatter(longitudes, latitudes, marker='.', color="green", s=1.0, zorder=2, alpha=0.0, transform=ccrs.PlateCarree())
lc = plot_utils.get_line_collection(coarse_ocean_mesh_graph, linewidths=0.16, colors="green", transform=ccrs.Geodetic())
ax.add_collection(lc)
if save_figures:
  plt.savefig(output_dir / 'coarse_ocean_large.png', bbox_inches='tight', transparent=True)


In [ ]:
fine_ocean_mesh = ocean_mesh_utils.get_ocean_mesh(domain=coarse_domain,
                                                  mesh_size=mesh_size,
                                                  save_mesh=(output_dir / 'fine_ocean_mesh.msh') if save_mesh else None,
                                                  min_size=100_000.0,
                                                  max_size=200_000.0)
fine_ocean_mesh_graph = mesh_graph.mesh_to_wgs(fine_ocean_mesh)

In [ ]:
fig = plt.figure(dpi=300)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mercator())
ax.set_extent(bounds)
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black'))
ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.4, color='k', linestyle='--')
lc = plot_utils.get_line_collection(fine_ocean_mesh_graph, linewidths=0.16, colors="green", transform=ccrs.Geodetic())
ax.add_collection(lc)
latitudes, longitudes = fine_ocean_mesh_graph.vertices
ax.scatter(longitudes, latitudes, marker='.', color="green", s=2.0, zorder=2, transform=ccrs.PlateCarree())

if save_figures:
    plt.savefig(output_dir / 'fine_ocean_mesh_zoomin.png', bbox_inches='tight')